# ATAG *Waypoint 2050* — non-CO₂ uncertainty on the as-reported scenarios

The reports quantify CO₂. AeroMAPS computes the non-CO₂ terms too, but every scenario in this
reproduction has so far run on a single central set of climate coefficients, so nothing published
here shows how wide the non-CO₂ answer actually is.

This notebook puts a band on the **as-reported scenarios** — the ones with no contrail mitigation
at all (`operations_contrails_start_year = 2101`). Contrail *avoidance* is a separate question and
is quantified in `climate_analysis.ipynb`.

Two distinct uncertainties are represented, and they are not the same question:

1. **How large the non-CO₂ effect is** — the contrail radiative-forcing sensitivity.
2. **How much decarbonisation reduces it** — cleaner fuels emit fewer soot particles, which
   reduces contrail forcing, but the modelling literature disagrees by a factor of three on how
   much.

Bands are named by **climate impact**: `High` is the worst outcome on *both* axes at once, `Low`
the best. Definitions and provenance are in `non_co2_uncertainty.yaml`.

In [ ]:
# The scenario ships with the package; copy it somewhere writable before running,
# so this notebook's outputs and regenerated inputs land in ./workdir rather than
# in the installed AeroMAPS.
from aeromaps.utils.scenarios import prepare_scenario
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aeromaps import assemble_processes, create_process

import utils
from utils import BANDS, apply_non_co2_band, band_name, load_non_co2_bands

SCENARIO = prepare_scenario("atag_climate_analysis")

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})

reference, bands, pairing = load_non_co2_bands()
print(reference["citation"].strip())
print("dois:", ", ".join(reference["dois"]))
print()
print(
    f"{'band':9s} {'sensitivity_rf':>16s} {'SAF EIn':>12s} {'implied SAF contrail reduction':>32s}"
)
for key in BANDS:
    b = bands[key]
    print(
        f"{b['label']:9s} {b['sensitivity_rf']:16.3e} "
        f"{b['saf_emission_index_particles_number']:12.4g} "
        f"{b['implied_saf_contrail_reduction_percent']:30.1f} %"
    )

## Running the scenarios under each band

Three scenarios — the document's set — each under the three bands. The band is applied to an
already-built process: the contrail sensitivity goes to the climate model's species settings, the
particle-number index to every non-default drop-in pathway.

In [ ]:
SCENARIOS = [
    ("3rd_edition_light", str(SCENARIO / "config_files" / "config_s0.yaml"), "s0", "S0 reference"),
    ("3rd_edition_full", str(SCENARIO / "config_files" / "config_s1.yaml"), "s1", "S1 SAF-focused"),
    (
        "3rd_edition_full",
        str(SCENARIO / "config_files" / "config_s2.yaml"),
        "s2",
        "S2 technology-centric",
    ),
]

runs = {}  # (scenario_label, band_key) -> process
saf_applied = {}
cwd = os.getcwd()
try:
    for edition, config, _scenario, label in SCENARIOS:
        os.chdir(utils.BASE / edition)
        for key in BANDS:
            process = create_process(config)
            saf_applied[label] = apply_non_co2_band(process, bands[key])
            process.compute()
            runs[(label, key)] = process
finally:
    os.chdir(cwd)

print(f"{len(runs)} runs across {len(SCENARIOS)} scenarios x {len(BANDS)} bands")
for label, pathways in saf_applied.items():
    print(f"  {label:24s} band applied to {len(pathways)} SAF pathway(s)")

### Guards

Four checks, in increasing subtlety. The last two are the ones that would fail silently.

In [ ]:
T_TOTAL = "temperature_increase_from_aviation"
T_CONTRAILS = "temperature_increase_from_contrails_from_aviation"

# 1. Every band's CENTRAL must reproduce the committed scenario. Both axes are left at their
#    packaged values there, so this is the same 1e-3 drift bar the other notebooks use (the
#    committed outputs were generated on Linux; the aerosol chain differs at ~1e-5 elsewhere).
for edition, _config, scenario, label in SCENARIOS:
    committed = json.loads(
        (utils.BASE / edition / "data_outputs" / f"{scenario}.json").read_text()
    )["climate_outputs"]
    got = runs[(label, "central")].data["climate_outputs"]
    worst = max(
        np.nanmax(np.abs(np.array(committed[k], dtype=float) - got[k].to_numpy(dtype=float)))
        / max(np.nanmax(np.abs(np.array(committed[k], dtype=float))), 1e-30)
        for k in committed
    )
    print(f"  {label:24s} central vs committed {scenario}.json: {worst:.2e}")
    assert worst < 1e-3, f"{label} central does not reproduce the committed scenario"

# 2. Bands must be monotone in warming, in every year - not just at 2050.
for _e, _c, _s, label in SCENARIOS:
    series = {k: runs[(label, k)].data["climate_outputs"][T_TOTAL] for k in BANDS}
    years = series["central"].index
    ok = (series["low"] <= series["central"] + 1e-12).all() and (
        series["central"] <= series["high"] + 1e-12
    ).all()
    print(f"  {label:24s} low <= central <= high in all {len(years)} years: {ok}")
    assert ok, f"{label} bands are not monotone"

# 3. NEITHER axis touches CO2, so co2_emissions must be bit-identical across the three bands.
#    If this ever fails, a band is reaching something it should not.
for _e, _c, _s, label in SCENARIOS:
    ref = runs[(label, "central")].data["climate_outputs"]["co2_emissions"]
    worst = max(
        float((runs[(label, k)].data["climate_outputs"]["co2_emissions"] - ref).abs().max())
        for k in BANDS
    )
    print(f"  {label:24s} co2_emissions identical across bands: max |diff| = {worst:.3e}")
    assert worst == 0.0, f"{label} bands moved CO2, which they must not"

print("\nall guards passed")

### The bands do not leak between processes

`apply_non_co2_band` deep-copies the climate settings before touching them. Without that, the
first band would edit the dict every later process shares and the answer would depend on the order
the bands happened to run in — a failure that produces plausible numbers and no error. Running the
bands in reverse and comparing is the direct test.

In [ ]:
cwd = os.getcwd()
try:
    os.chdir(utils.BASE / "3rd_edition_full")
    reversed_runs = {}
    for key in reversed(BANDS):  # deliberately the other order
        process = create_process(str(SCENARIO / "config_files" / "config_s1.yaml"))
        apply_non_co2_band(process, bands[key])
        process.compute()
        reversed_runs[key] = process.data["climate_outputs"][T_TOTAL]
finally:
    os.chdir(cwd)

for key in BANDS:
    forward = runs[("S1 SAF-focused", key)].data["climate_outputs"][T_TOTAL]
    delta = float((reversed_runs[key] - forward).abs().max())
    print(f"  {bands[key]['label']:12s} forward vs reversed order: max |diff| = {delta:.3e}")
    assert delta == 0.0, "band results depend on execution order - the settings dict leaked"
print("\norder-independent")

## How wide is the band?

In [ ]:
rows = []
for _e, _c, _s, label in SCENARIOS:
    row = {"scenario": label}
    for key in BANDS:
        clim = runs[(label, key)].data["climate_outputs"]
        row[f"{bands[key]['label']} [mK]"] = 1000 * clim.loc[2050, T_TOTAL]
    lo = row[f"{bands['low']['label']} [mK]"]
    hi = row[f"{bands['high']['label']} [mK]"]
    mid = row[f"{bands['central']['label']} [mK]"]
    row["width [mK]"] = hi - lo
    row["width [% of central]"] = 100 * (hi - lo) / mid
    rows.append(row)

band_table = pd.DataFrame(rows).set_index("scenario").round(1)
display(band_table)

central = band_table[f"{bands['central']['label']} [mK]"]
spread_between = central.max() - central.min()
widest_band = band_table["width [mK]"].max()
print(f"spread BETWEEN scenarios (central band): {spread_between:.1f} mK")
print(f"widest band WITHIN a single scenario   : {widest_band:.1f} mK")
print(
    f"ratio: the non-CO2 uncertainty is {widest_band / spread_between:.1f}x the "
    f"difference between the scenarios"
)

## Which axis dominates?

The two axes are separable, and it is not obvious in advance which matters more. Holding one at
its central value while the other moves isolates each contribution — run here on S1.

In [ ]:
def s1_run(sensitivity_rf, ein_saf):
    """One S1 run with the two axes set independently."""
    band = dict(bands["central"])
    band["sensitivity_rf"] = sensitivity_rf
    band["saf_emission_index_particles_number"] = ein_saf
    process = create_process(str(SCENARIO / "config_files" / "config_s1.yaml"))
    apply_non_co2_band(process, band)
    process.compute()
    return 1000 * process.data["climate_outputs"].loc[2050, T_TOTAL]


c = bands["central"]
axis_rows = []
cwd = os.getcwd()
try:
    os.chdir(utils.BASE / "3rd_edition_full")
    for key in BANDS:
        b = bands[key]
        axis_rows.append(
            {
                "band": b["label"],
                "contrail sensitivity only": s1_run(
                    b["sensitivity_rf"], c["saf_emission_index_particles_number"]
                ),
                "SAF benefit only": s1_run(
                    c["sensitivity_rf"], b["saf_emission_index_particles_number"]
                ),
                "both (as bundled)": 1000
                * runs[("S1 SAF-focused", key)].data["climate_outputs"].loc[2050, T_TOTAL],
            }
        )
finally:
    os.chdir(cwd)

axes = pd.DataFrame(axis_rows).set_index("band").round(1)
display(axes)

for column in axes.columns:
    span = axes[column].max() - axes[column].min()
    print(f"  {column:28s} spans {span:6.1f} mK")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.6))
labels = [bands[k]["label"] for k in BANDS]
x = np.arange(len(labels))
width = 0.26
for offset, column, colour in zip(
    (-width, 0.0, width),
    ("contrail sensitivity only", "SAF benefit only", "both (as bundled)"),
    ("#4C72B0", "#DD8452", "#55A868"),
):
    ax.bar(x + offset, axes[column].to_numpy(), width, label=column, color=colour)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("2050 warming from aviation [mK]")
ax.set_title("S1: contribution of each uncertainty axis")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.show()

## Temperature and forcing under the band

In [ ]:
envelope_processes = {}
groups = {}
middle = {}
for _e, _c, _s, label in SCENARIOS:
    members = []
    for key in BANDS:
        name = band_name(label, bands[key]["label"])
        envelope_processes[name] = runs[(label, key)]
        members.append(name)
    groups[label] = members
    middle[label] = band_name(label, bands["central"]["label"])

banded = assemble_processes(envelope_processes)
print(f"{len(envelope_processes)} scenarios in {len(groups)} envelopes")

### Persist a tidy summary

The document reads committed outputs only and runs no models, the same rule the sweep follows. Since these are cross-band comparisons rather than single scenario files, the result is a tidy CSV (mirroring `sweep_results.csv.gz`) rather than one JSON per run.

In [ ]:
COLUMNS = [T_TOTAL, T_CONTRAILS, "non_co2_erf", "contrails_erf"]

tidy_rows = []
for _e, _c, _s, label in SCENARIOS:
    for key in BANDS:
        clim = runs[(label, key)].data["climate_outputs"]
        for year in clim.index:
            row = {"scenario": label, "band": bands[key]["label"], "band_key": key, "year": year}
            for column in COLUMNS:
                row[column] = clim.loc[year, column]
            tidy_rows.append(row)

tidy = pd.DataFrame(tidy_rows)
tidy.to_csv("baseline_uncertainty_results.csv.gz", index=False)
print(
    f"wrote baseline_uncertainty_results.csv.gz: {len(tidy)} rows, "
    f"{tidy['scenario'].nunique()} scenarios x {tidy['band'].nunique()} bands"
)

In [ ]:
banded.plot(
    "temperature_increase_comparison",
    scenario_groups=groups,
    group_display="envelope",
    group_envelope_show_members=False,
    group_envelope_middle=middle,
)

In [ ]:
banded.plot(
    "contrails_temperature_comparison",
    scenario_groups=groups,
    group_display="envelope",
    group_envelope_show_members=False,
    group_envelope_middle=middle,
)

In [ ]:
banded.plot(
    "non_co2_erf_comparison",
    scenario_groups=groups,
    group_display="envelope",
    group_envelope_show_members=False,
    group_envelope_middle=middle,
)

## Against an external estimate

The ICCT's *Aviation Vision 2050* (September 2025) reports aviation warming over **2025–2050** in
millikelvin, the same quantity and units AeroMAPS emits, which makes it the one directly
comparable external number available. Its historical-trends case adds 60 mK over that window.

The comparison is indicative, not a validation: the scenario definitions, traffic forecasts and
climate model all differ. What it establishes is whether this reproduction lands in the same
range.

In [ ]:
ICCT_HISTORICAL_TRENDS_MK = 60.0  # ICCT Aviation Vision 2050, added warming 2025-2050

rows = []
for _e, _c, _s, label in SCENARIOS:
    row = {"scenario": label}
    for key in BANDS:
        clim = runs[(label, key)].data["climate_outputs"][T_TOTAL]
        row[bands[key]["label"]] = 1000 * (clim.loc[2050] - clim.loc[2025])
    rows.append(row)

added = pd.DataFrame(rows).set_index("scenario").round(1)
added.columns = [f"{c} [mK]" for c in added.columns]
print("Added warming 2025-2050:")
display(added)
print(f"ICCT historical trends, same window: {ICCT_HISTORICAL_TRENDS_MK:.0f} mK")
print()
print("S0 is the closest analogue to a no-further-action case; S1 and S2 include the reports'")
print("own mitigation, so they should sit below the ICCT historical-trends figure.")

In [ ]:
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests()